In [30]:
# predict_4sites_no_osmnx.py

import pickle
import numpy as np
import pandas as pd
import networkx as nx
import math

from datetime import datetime, date, timedelta
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler

# ─── CONFIG ──────────────────────────────────────────────────────────────
TRAFFIC_CSV   = 'traffic_cleaned.csv'
MODEL_PATH    = 'models/lstm_model_all.h5'
SCALER_PATH   = 'models/scaler_all.pkl'
SEQ_LEN       = 24
NUM_ROUTES    = 5
INTER_DELAY   = 30       # seconds per intersection
FREE_SPEED    = 60.0     # km/h
# ─────────────────────────────────────────────────────────────────────────

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # km
    φ1, λ1, φ2, λ2 = map(math.radians, (lat1, lon1, lat2, lon2))
    dφ = φ2 - φ1; dλ = λ2 - λ1
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return R * 2*math.atan2(math.sqrt(a), math.sqrt(1-a))

def flow_to_time(flow, dist_km):
    if flow < 300:
        speed = FREE_SPEED
    else:
        speed = max(5.0, FREE_SPEED*(1-(flow-300)/1000.0))
    return dist_km/speed + INTER_DELAY/3600.0  # hours

def load_model_and_scaler():
    model  = load_model(MODEL_PATH, compile=False)
    scaler = pickle.load(open(SCALER_PATH,'rb'))
    return model, scaler

def load_sites():
    df = pd.read_csv(TRAFFIC_CSV, parse_dates=['DateTime'])
    df = df.rename(columns=lambda c:c.strip()).rename(columns={
        'SCATS Number':'scats',
        'NB_LATITUDE':'lat',
        'NB_LONGITUDE':'lon'
    })
    site_ids = df['scats'].unique()[:4]
    sites = (df[df['scats'].isin(site_ids)]
               [['scats','lat','lon']]
               .drop_duplicates('scats')
               .set_index('scats'))
    return df, sites

def forecast_flows(df, model, scaler, depart_dt, site_ids):
    flows = {}
    for sc in site_ids:
        ts = (df[df['scats']==sc]
                .set_index('DateTime')['Traffic_flow']
                .sort_index())
        cutoff = depart_dt - timedelta(hours=1)
        if cutoff < ts.index[0] or cutoff > ts.index[-1]:
            hist = ts.iloc[-SEQ_LEN:].values.reshape(-1,1)
        else:
            hist = ts.loc[:cutoff].iloc[-SEQ_LEN:].values.reshape(-1,1)
        if len(hist)<SEQ_LEN:
            hist = np.pad(hist, ((SEQ_LEN-len(hist),0),(0,0)), 'constant')
        scaled = scaler.transform(hist)
        inp    = scaled.reshape(1,SEQ_LEN,1).astype(np.float32)
        pred_s = model.predict(inp, verbose=0)
        flows[sc] = scaler.inverse_transform(pred_s)[0,0]
    return flows

def suggest_routes(origin, dest, depart_dt):
    # load
    model, scaler = load_model_and_scaler()
    df, sites     = load_sites()
    site_ids      = list(sites.index)

    # forecast flows
    flows = forecast_flows(df, model, scaler, depart_dt, site_ids)

    # build complete digraph
    G = nx.DiGraph()
    for u in site_ids:
        for v in site_ids:
            if u==v: continue
            lat1, lon1 = sites.at[u,'lat'], sites.at[u,'lon']
            lat2, lon2 = sites.at[v,'lat'], sites.at[v,'lon']
            d_km = haversine(lat1,lon1,lat2,lon2)
            t_h  = flow_to_time(flows[v], d_km)
            G.add_edge(u, v, distance_km=d_km, time_h=t_h)

    # if same
    if origin==dest:
        return [([origin],0.0, 0.0)]

    # find up to NUM_ROUTES simple paths by time
    paths = nx.shortest_simple_paths(G, origin, dest, weight='time_h')
    seen, routes = set(), []
    for path in paths:
        r_tuple = tuple(path)
        if r_tuple in seen: continue
        seen.add(r_tuple)
        total_h = sum(G[u][v]['time_h'] for u,v in zip(path,path[1:]))
        total_km= sum(G[u][v]['distance_km'] for u,v in zip(path,path[1:]))
        routes.append((path, total_h*60, total_km))
        if len(routes)>=NUM_ROUTES: break
    return routes

if __name__ == '__main__':
    origin = int(input("Origin SCATS #: "))
    dest   = int(input("Destination SCATS #: "))
    hhmm   = input("Departure time (HH:MM): ")
    depart_dt = datetime.combine(date.today(),
                     datetime.strptime(hhmm, "%H:%M").time())

    # load sites_df here so we can use it for coords
    df, sites_df = load_sites()
    out = suggest_routes(origin, dest, depart_dt)

    print(f"\nTop {len(out)} routes from {origin}→{dest} at {hhmm}:\n")
    for i, (route, tmin, dist_km) in enumerate(out, 1):
        coord_str = " → ".join(
            f"({sites_df.at[s,'lat']:.4f}, {sites_df.at[s,'lon']:.4f})"
            for s in route
        )
        print(f"Route {i}: {route} → {coord_str}")
        print(f"  • Time: {tmin:.1f} min, Distance: {dist_km:.2f} km\n")



Origin SCATS #:  970
Destination SCATS #:  2200
Departure time (HH:MM):  08:40



Top 5 routes from 970→2200 at 08:40:

Route 1: [970, 2200] → (-37.8670, 145.0916) → (-37.8163, 145.0981)
  • Time: 6.2 min, Distance: 5.67 km

Route 2: [970, 2000, 2200] → (-37.8670, 145.0916) → (-37.8517, 145.0943) → (-37.8163, 145.0981)
  • Time: 6.7 min, Distance: 5.67 km

Route 3: [970, 2820, 2200] → (-37.8670, 145.0916) → (-37.7948, 145.0308) → (-37.8163, 145.0981)
  • Time: 17.0 min, Distance: 16.03 km

Route 4: [970, 2000, 2820, 2200] → (-37.8670, 145.0916) → (-37.8517, 145.0943) → (-37.7948, 145.0308) → (-37.8163, 145.0981)
  • Time: 18.0 min, Distance: 16.55 km

Route 5: [970, 2820, 2000, 2200] → (-37.8670, 145.0916) → (-37.7948, 145.0308) → (-37.8517, 145.0943) → (-37.8163, 145.0981)
  • Time: 23.5 min, Distance: 22.04 km



In [36]:
# predict_no_csv.py

import pickle
import numpy as np
import networkx as nx
import math

from datetime import datetime, date, timedelta
from tensorflow.keras.models import load_model

# ─── CONFIG ──────────────────────────────────────────────────────────────
MODEL_PATH    = 'models/gru_model_all.h5'      # or whichever you trained
SCALER_PATH   = 'models/scaler_all.pkl'
SITES_PKL     = 'models/sites_data.pkl'
SEQ_LEN       = 24
NUM_ROUTES    = 5
INTER_DELAY   = 30       # seconds per intersection
FREE_SPEED    = 60.0     # km/h
# ─────────────────────────────────────────────────────────────────────────

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # km
    φ1, λ1, φ2, λ2 = map(math.radians, (lat1, lon1, lat2, lon2))
    dφ = φ2 - φ1; dλ = λ2 - λ1
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return R * 2*math.atan2(math.sqrt(a), math.sqrt(1-a))

def flow_to_time(flow, dist_km):
    speed = FREE_SPEED if flow < 300 else max(5.0, FREE_SPEED*(1-(flow-300)/1000.0))
    return dist_km/speed + INTER_DELAY/3600.0  # hours

# 1) Load model, scaler, and sites_data
model  = load_model(MODEL_PATH, compile=False)
scaler = pickle.load(open(SCALER_PATH,'rb'))
sd     = pickle.load(open(SITES_PKL,'rb'))
sites_df  = sd['sites_df']    # DataFrame indexed by scats
flow_hist = sd['flow_hist']   # dict: scats → pd.Series of flows

def forecast_flows(depart_dt):
    flows = {}
    for sc, ts in flow_hist.items():
        cutoff = depart_dt - timedelta(hours=1)
        if cutoff < ts.index[0] or cutoff > ts.index[-1]:
            hist = ts.iloc[-SEQ_LEN:].values.reshape(-1,1)
        else:
            hist = ts.loc[:cutoff].iloc[-SEQ_LEN:].values.reshape(-1,1)
        if len(hist) < SEQ_LEN:
            hist = np.pad(hist, ((SEQ_LEN-len(hist),0),(0,0)), 'constant')
        s = scaler.transform(hist)
        inp = s.reshape(1, SEQ_LEN, 1).astype(np.float32)
        pred = model.predict(inp, verbose=0)
        flows[sc] = scaler.inverse_transform(pred)[0,0]
    return flows

def suggest_routes(origin, dest, hhmm):
    depart_dt = datetime.combine(date.today(), datetime.strptime(hhmm,"%H:%M").time())
    flows = forecast_flows(depart_dt)

    # build fully-connected graph
    G = nx.DiGraph()
    for u in sites_df.index:
        for v in sites_df.index:
            if u==v: continue
            lat1, lon1 = sites_df.at[u,'lat'], sites_df.at[u,'lon']
            lat2, lon2 = sites_df.at[v,'lat'], sites_df.at[v,'lon']
            d_km = haversine(lat1,lon1,lat2,lon2)
            t_h  = flow_to_time(flows[v], d_km)
            G.add_edge(u, v, time_h=t_h, distance_km=d_km)

    if origin==dest:
        return [( [origin], 0.0, 0.0 )]

    paths, seen, out = nx.shortest_simple_paths(G, origin, dest, weight='time_h'), set(), []
    for path in paths:
        tup = tuple(path)
        if tup in seen: continue
        seen.add(tup)
        total_h  = sum(G[u][v]['time_h'] for u,v in zip(path,path[1:]))
        total_km = sum(G[u][v]['distance_km'] for u,v in zip(path,path[1:]))
        out.append((list(path), total_h*60, total_km))
        if len(out) >= NUM_ROUTES: break
    return out

# ─── CLI ────────────────────────────────────────────────────────────────
if __name__=='__main__':
    origin = int(input("Origin SCATS #: "))
    dest   = int(input("Destination SCATS #: "))
    hhmm   = input("Departure time (HH:MM): ")

    routes = suggest_routes(origin, dest, hhmm)
    print(f"\nTop {len(routes)} routes from {origin}→{dest} at {hhmm}:\n")
    for i,(r,tmin,dist) in enumerate(routes,1):
        coords = " → ".join(f"({sites_df.at[s,'lat']:.4f},{sites_df.at[s,'lon']:.4f})" for s in r)
        print(f"Route {i}: {r} → {coords}")
        print(f"  • Time: {tmin:.1f} min, Distance: {dist:.2f} km\n")


Origin SCATS #:  2200
Destination SCATS #:  2000
Departure time (HH:MM):  10:20



Top 5 routes from 2200→2000 at 10:20:

Route 1: [2200, 2000] → (-37.8163,145.0981) → (-37.8517,145.0943)
  • Time: 4.4 min, Distance: 3.95 km

Route 2: [2200, 3682, 2000] → (-37.8163,145.0981) → (-37.8370,145.0970) → (-37.8517,145.0943)
  • Time: 5.0 min, Distance: 3.95 km

Route 3: [2200, 3126, 2000] → (-37.8163,145.0981) → (-37.8278,145.0988) → (-37.8517,145.0943)
  • Time: 5.0 min, Distance: 3.96 km

Route 4: [2200, 3126, 3682, 2000] → (-37.8163,145.0981) → (-37.8278,145.0988) → (-37.8370,145.0970) → (-37.8517,145.0943)
  • Time: 5.5 min, Distance: 3.96 km

Route 5: [2200, 3685, 2000] → (-37.8163,145.0981) → (-37.8547,145.0938) → (-37.8517,145.0943)
  • Time: 5.6 min, Distance: 4.62 km

